# Лабораторная работа 4. Линейная классификация: логистическая регрессия, метрики и SVM

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 3 |
| Опора на лекции | лекция 3: сигмоида (опр. 3.1), логистическая потеря (опр. 3.3), отступ (опр. 3.8), прямая и двойственная задачи SVM (опр. 3.9, теорема 3.11), опорные векторы, мягкий зазор (опр. 3.14), ядра (опр. 3.17) и критерий Мерсера |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Понять, чем логистическая регрессия и SVM отличаются друг от друга — и увидеть, что различие целиком заключено в виде функции потерь. Научиться измерять качество классификации так, чтобы не обмануть себя: разобрать, когда accuracy бесполезна, а ROC-AUC оптимистична.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab04_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=4)
describe_variant(variant)

---
# Часть 1. Три потери как функции отступа

Определение 3.3 задаёт логистическую потерю при $y\in\{0,1\}$. Если перейти к
разметке $y\in\{-1,+1\}$ и ввести **отступ** $M = y\,\theta^{\mathsf T}x$, то она
записывается как $\ln(1+e^{-M})$, и все три потери курса становятся сравнимы:

$$
[M<0] \quad\text{(пороговая)},\qquad
\ln(1+e^{-M}) \quad\text{(логистическая)},\qquad
\max(0,\,1-M) \quad\text{(hinge, SVM)}.
$$

Отступ — это «уверенность с учётом правильности»: $M>0$ означает верный ответ,
и чем больше $M$, тем дальше объект от границы.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def sigmoid(z):
    """Численно устойчивая сигмоида: без переполнения при больших |z|.

    Наивная формула 1/(1+exp(-z)) при z = -1000 даёт exp(1000) = inf.
    Приём: для z >= 0 считать 1/(1+exp(-z)), для z < 0 -- exp(z)/(1+exp(z)).
    """
    # TODO (4-5 строк)
    raise NotImplementedError


print("sigmoid(-1000) =", sigmoid(np.array([-1000.0]))[0], "-- без переполнения")

In [ ]:
M = np.linspace(-3, 3, 400)
fig, ax = plt.subplots()
ax.plot(M, (M < 0).astype(float), lw=2, label=r"пороговая $[M<0]$")
ax.plot(M, np.log(1 + np.exp(-M)), lw=2, label=r"логистическая $\ln(1+e^{-M})$")
ax.plot(M, np.maximum(0, 1 - M), lw=2, label=r"hinge $\max(0,\,1-M)$")
ax.set_xlabel(r"отступ $M = y\,\theta^{\mathsf{T}}x$"); ax.set_ylabel("потеря")
ax.set_title("Потери как функции отступа"); ax.legend()
plt.tight_layout(); plt.show()

> **Вывод.** Почему пороговую потерю не минимизируют напрямую? Какие два свойства логистической и hinge-потерь делают их пригодными?
>
> *(ваш ответ здесь)*

---
# Часть 2. Разделимая выборка: веса уходят в бесконечность

У логистической потери на линейно разделимой выборке **нет минимума**: для любого
$\theta$, разделяющего классы, увеличение $\|\theta\|$ в $c$ раз увеличивает все
отступы и уменьшает $Q$. Формально $\inf_\theta Q = 0$, но инфимум не достигается.

In [ ]:
X_sep = rng.normal(size=(60, 2))
y_sep = (X_sep[:, 0] + X_sep[:, 1] > 0).astype(int)      # заведомо разделимо

rows = []
for C in [0.01, 1.0, 100.0, 10_000.0]:
    m = LogisticRegression(C=C, max_iter=100_000).fit(X_sep, y_sep)
    p = m.predict_proba(X_sep)[:, 1]
    rows.append({"C": C, "||theta||": np.linalg.norm(m.coef_),
                 "мин. вероятность": p.min(), "макс. вероятность": p.max(),
                 "ошибок": int((m.predict(X_sep) != y_sep).sum())})
display(pd.DataFrame(rows).set_index("C").round(4))
print("C -- обратный параметр регуляризации: чем больше C, тем слабее штраф")

> **Вывод.** Что происходит с $\|\theta\|$ и с вероятностями при росте $C$? Почему это плохо, даже если все ответы верны?
>
> *(ваш ответ здесь)*

---
# Часть 3. Метрики: где легко обмануть себя

Матрица ошибок при пороге $t$ даёт четыре числа, из которых собираются все
метрики:

$$
\text{precision} = \frac{TP}{TP+FP},\quad
\text{recall} = TPR = \frac{TP}{TP+FN},\quad
FPR = \frac{FP}{FP+TN}.
$$

ROC-кривая — точки $(FPR(t), TPR(t))$ по всем порогам; PR-кривая —
$(\text{recall}, \text{precision})$. Посмотрим, что происходит с ними при
росте дисбаланса классов.

> **Напоминание — матрица ошибок, $F_1$ и AUC.** Модель выдаёт балл, порог $t$ превращает его в ответ, и каждый объект попадает
> в одну из четырёх клеток:
>
> | | предсказано 1 | предсказано 0 |
> |---|---|---|
> | **на самом деле 1** | $TP$ (верно найден) | $FN$ (пропуск) |
> | **на самом деле 0** | $FP$ (ложная тревога) | $TN$ (верно отвергнут) |
>
> Отсюда: *precision* — какая доля тревог оказалась настоящей, *recall* (он же
> $TPR$, полнота) — какую долю настоящих единиц мы нашли, $FPR$ — какую долю
> нулей зря объявили единицами. Precision и recall тянут в разные стороны:
> объявив всех единицами, получим recall $= 1$ при мизерной precision.
> $F_1 = 2\,\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$ —
> их гармоническое среднее: оно мало, если мала хотя бы одна из величин.
>
> $AUC$ — площадь под кривой; для ROC она равна вероятности того, что случайный
> объект класса 1 получит балл выше случайного объекта класса 0 (проверим это
> в домашней работе). У случайной модели ROC-AUC $= 0.5$, у идеальной — $1$.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score


def evaluate_at_balance(frac_pos, seed=RANDOM_STATE):
    """Обучить логрег на выборке с заданной долей класса 1 и вернуть метрики."""
    X, y = make_classification(n_samples=4000, n_features=20, n_informative=5,
                              n_redundant=3, class_sep=0.9, flip_y=0.02,
                              weights=[1 - frac_pos, frac_pos], random_state=seed)
    Xa, Xv, ya, yv = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    s = StandardScaler().fit(Xa)
    p = LogisticRegression(max_iter=5000).fit(s.transform(Xa), ya).predict_proba(
        s.transform(Xv))[:, 1]
    return {"доля класса 1": yv.mean(),
            "accuracy «всегда 0»": accuracy_score(yv, np.zeros_like(yv)),
            "accuracy модели": accuracy_score(yv, (p >= 0.5).astype(int)),
            "ROC-AUC": roc_auc_score(yv, p), "PR-AUC": average_precision_score(yv, p)}

In [ ]:
tab = pd.DataFrame([evaluate_at_balance(f)
                    for f in [0.5, 0.35, 0.2, 0.1, 0.05, 0.03]]).set_index("доля класса 1")
display(tab.round(3))

fig, ax = plt.subplots()
for col in tab.columns:
    ax.plot(tab.index, tab[col], "o-", lw=2, label=col)
ax.invert_xaxis(); ax.set_xlabel("доля класса 1"); ax.set_ylabel("значение метрики")
ax.set_title("Что происходит с метриками при росте дисбаланса"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

> **Вывод.** Какая метрика перестаёт различать модель и константу? Чем PR-AUC отличается от ROC-AUC и какую вы возьмёте для поиска редкого события (1 % положительных)?
>
> *(ваш ответ здесь)*

### Задание 3.1. Порог — это решение, а не константа

Порог 0.5 ниоткуда не следует. Постройте precision, recall и $F_1$ как функции
порога и найдите порог, максимизирующий $F_1$.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

X, y = make_classification(n_samples=4000, n_features=20, n_informative=5,
                          class_sep=0.9, flip_y=0.02, weights=[0.9, 0.1],
                          random_state=RANDOM_STATE)
Xa, Xv, ya, yv = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
s = StandardScaler().fit(Xa)
scores = LogisticRegression(max_iter=5000).fit(s.transform(Xa), ya).predict_proba(
    s.transform(Xv))[:, 1]

ts = np.linspace(0.02, 0.98, 200)
# TODO (3-4 строки): посчитайте precision, recall и F1 для каждого порога,
#                    найдите порог с максимальным F1 и сравните с порогом 0.5.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

fpr, tpr, _ = roc_curve(yv, scores)
prec, rec, _ = precision_recall_curve(yv, scores)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(yv, scores):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="случайный ответ")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC")
axes[0].legend()

axes[1].plot(rec, prec, lw=2, label=f"AP = {average_precision_score(yv, scores):.3f}")
axes[1].axhline(yv.mean(), ls="--", color="black", label=f"доля класса 1 = {yv.mean():.2f}")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_title("PR")
axes[1].legend()

axes[2].plot(ts, P, lw=2, label="precision"); axes[2].plot(ts, R, lw=2, label="recall")
axes[2].plot(ts, F1, lw=2, label="$F_1$")
axes[2].axvline(t_best, ls="--", color="black", label=f"$t^* = {t_best:.2f}$")
axes[2].set_xlabel("порог"); axes[2].set_title("Метрики против порога"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

---
# Часть 4. SVM: решение зависит только от опорных векторов

Прямая задача (опр. 3.9): $\min\frac12\|w\|^2$ при $y_i(w^{\mathsf T}x_i+b)\ge1$.
В двойственной задаче (теорема 3.11) появляются множители $\alpha_i$, и

$$
w^* = \sum_i \alpha_i^* y_i x_i .
$$

Объекты с $\alpha_i^*>0$ — **опорные векторы**; по условию дополняющей нежёсткости
они лежат ровно на границе полосы. Все остальные в сумму не входят вовсе.
Проверим это буквально: удалим их.

> **Напоминание — условие дополняющей нежёсткости.** Это одно из условий Каруша–Куна–Таккера — необходимых условий минимума при
> ограничениях-неравенствах. Каждому ограничению $g_i(w)\le0$ отвечает множитель
> $\alpha_i\ge0$, и в точке минимума обязательно $\alpha_i\,g_i(w) = 0$: либо
> множитель нулевой, либо ограничение выполнено **как равенство**.
>
> Для SVM ограничение — это $y_i(w^{\mathsf T}x_i + b)\ge1$, поэтому:
> объект строго вне полосы ($>1$) $\Rightarrow$ $\alpha_i = 0$ и в решении он не
> участвует; объект с $\alpha_i>0$ $\Rightarrow$ отступ ровно $1$, то есть он
> лежит на границе полосы. Отсюда и всё поведение метода: решение определяется
> горсткой пограничных объектов, а не всей выборкой.

In [ ]:
Xb, yb = make_blobs(n_samples=60, centers=2, cluster_std=1.0, random_state=RANDOM_STATE)
yb = np.where(yb == 0, -1, 1)
Xb = StandardScaler().fit_transform(Xb) * 1.2

clf = SVC(kernel="linear", C=1e6).fit(Xb, yb)          # C огромно => жёсткий зазор
sv = clf.support_
print(f"объектов {len(yb)}, опорных векторов {len(sv)}")
print(f"w = {np.round(clf.coef_[0], 5)}, b = {clf.intercept_[0]:.5f}")
print(f"ширина полосы 2/||w|| = {2 / np.linalg.norm(clf.coef_[0]):.4f}")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

# TODO (2-3 строки): обучите SVC(kernel="linear", C=1e6) ТОЛЬКО на опорных
#   векторах Xb[sv], yb[sv] и сравните полученные w и b с исходными.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(*Xb[yb == 1].T, s=35, marker="o", label="$y = +1$")
ax.scatter(*Xb[yb == -1].T, s=35, marker="s", label="$y = -1$")
ax.scatter(*Xb[sv].T, s=180, facecolors="none", edgecolors="red", lw=1.6,
           label=f"опорные векторы ({len(sv)})")
gx = np.linspace(Xb[:, 0].min() - 0.5, Xb[:, 0].max() + 0.5, 200)
w, b = clf.coef_[0], clf.intercept_[0]
for level, style in [(0, "-"), (1, "--"), (-1, "--")]:
    ax.plot(gx, -(w[0] * gx + b - level) / w[1], style, color="black", lw=1.5)
ax.set_ylim(Xb[:, 1].min() - 0.7, Xb[:, 1].max() + 0.7)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.legend(fontsize=8)
ax.set_title("Разделяющая полоса и опорные векторы")
plt.tight_layout(); plt.show()

In [ ]:
# Мягкий зазор (опр. 3.14): параметр C -- цена нарушения отступа
Xm, ym = make_moons(n_samples=200, noise=0.28, random_state=RANDOM_STATE)
ym = np.where(ym == 0, -1, 1)
Xm = StandardScaler().fit_transform(Xm)

rows = []
for C in [0.01, 0.1, 1.0, 100.0]:
    m = SVC(kernel="linear", C=C).fit(Xm, ym)
    rows.append({"C": C, "опорных векторов": len(m.support_),
                 "ширина полосы 2/||w||": 2 / np.linalg.norm(m.coef_[0])})
display(pd.DataFrame(rows).set_index("C").round(4))

> **Вывод.** Изменилось ли решение после удаления неопорных объектов? Как число опорных векторов зависит от $C$ и какому пределу отвечает $C\to\infty$?
>
> *(ваш ответ здесь)*

---
# Часть 5. Ядра

Определение 3.17: $K(x,x') = \langle\varphi(x),\varphi(x')\rangle$ для некоторого
отображения $\varphi$. Критерий Мерсера: $K$ — ядро тогда и только тогда, когда
матрица Грама $\|K(x_i,x_j)\|$ симметрична и неотрицательно определена.

Проверить это можно прямо: посчитать собственные числа матрицы Грама.

In [ ]:
pts = rng.normal(size=(40, 2))
d2 = np.sum(pts ** 2, 1)[:, None] - 2 * pts @ pts.T + np.sum(pts ** 2, 1)[None, :]

candidates = {
    "линейное  x^T x'": pts @ pts.T,
    "полиномиальное (x^T x' + 1)^3": (pts @ pts.T + 1) ** 3,
    "RBF exp(-||x-x'||^2)": np.exp(-np.maximum(d2, 0)),
    "сигмоидное tanh(0.1 x^T x')": np.tanh(0.1 * pts @ pts.T),
    "НЕ ядро: ||x - x'||": np.sqrt(np.maximum(d2, 0)),
}
for name, G in candidates.items():
    lam_min = np.linalg.eigvalsh((G + G.T) / 2).min()
    print(f"{name:32s} min lambda = {lam_min:+9.4f}  ->  "
          f"{'ядро' if lam_min > -1e-8 else 'НЕ ядро'}")

In [ ]:
# Ядровой переход: одна и та же двойственная задача, разные K
Xc, yc = make_circles(n_samples=200, noise=0.12, factor=0.45, random_state=RANDOM_STATE)
Xc = StandardScaler().fit_transform(Xc)


def plot_boundary(ax, model, X, y, title):
    g1, g2 = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    Z = model.decision_function(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
    ax.contourf(g1, g2, Z, levels=[-1e9, 0, 1e9], colors=["#cfe8e4", "#f6ddc4"], alpha=.8)
    ax.contour(g1, g2, Z, levels=[0], colors="black", linewidths=1.8)
    ax.scatter(*X[y == 0].T, s=14, marker="o"); ax.scatter(*X[y == 1].T, s=14, marker="s")
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, kw) in zip(axes, [("линейное", dict(kernel="linear")),
                                 ("полиномиальное d=2", dict(kernel="poly", degree=2)),
                                 ("RBF, gamma=1", dict(kernel="rbf", gamma=1.0)),
                                 ("RBF, gamma=100", dict(kernel="rbf", gamma=100.0))]):
    m = SVC(C=1.0, **kw).fit(Xc, yc)
    plot_boundary(ax, m, Xc, yc,
                  f"{name}\nточность {m.score(Xc, yc):.2f}, опорных {len(m.support_)}")
plt.tight_layout(); plt.show()

> **Вывод.** Какие функции не прошли критерий Мерсера? Что происходит с границей при $\gamma\to\infty$ и почему это переобучение?
>
> *(ваш ответ здесь)*

---
# Часть 6. Своя выборка

In [ ]:
data = load_personal(variant)
Xtr, Xte, ytr, yte = data["X_train"], data["X_test"], data["y_train"], data["y_test"]
if data["task"] == "regression":
    thr = np.median(ytr)
    ytr, yte = (ytr > thr).astype(int), (yte > thr).astype(int)
    print(f"регрессионный вариант бинаризован по медиане обучающей выборки")
ytr, yte = ytr.astype(int), yte.astype(int)

rows = []
for name, m in [("логистическая регрессия", LogisticRegression(C=1.0, max_iter=5000)),
                ("SVM (линейное ядро)", SVC(kernel="linear", C=1.0)),
                ("SVM (RBF)", SVC(kernel="rbf", C=1.0, gamma="scale"))]:
    m.fit(Xtr, ytr)
    s = m.decision_function(Xte)
    rows.append({"модель": name, "ROC-AUC": roc_auc_score(yte, s),
                 "accuracy": accuracy_score(yte, m.predict(Xte)),
                 "опорных векторов": len(m.support_) if isinstance(m, SVC) else "-"})
rows.append({"модель": "константа", "ROC-AUC": 0.5,
             "accuracy": max(yte.mean(), 1 - yte.mean()), "опорных векторов": "-"})
display(pd.DataFrame(rows).set_index("модель").round(4))

> **Вывод.** Какая модель победила? Сколько объектов оказалось опорными и о чём это говорит? Насколько корректно такое сравнение?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Логистическая регрессия минимизирует $\sum_i\ln(1+e^{-M_i})$, SVM — $\frac12\|w\|^2 + C\sum_i\max(0,1-M_i)$. Назовите два различия в поведении, следующих прямо из вида этих потерь.
2. Почему на линейно разделимой выборке у логистической регрессии без регуляризации нет минимума, а у SVM с жёстким зазором — есть?
3. У вас 1 000 000 объектов, из них 500 положительных. Модель даёт accuracy 0.9995 и ROC-AUC 0.97. Достаточно ли этого для внедрения?
4. Что такое опорный вектор и почему классификатор не меняется при удалении неопорных объектов?

---

**Дома:** откройте `lab04_homework.ipynb` — там три задачи: своя логистическая регрессия с методом Ньютона, своя ROC-кривая и двойственная задача SVM.